In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
############################################################
# C2 + C3 RE-RUN WITH FIXED PROMPT (C1 already done)
# Self-contained: install, load, run C2, run C3, save all
############################################################
!pip install -q sentence-transformers faiss-cpu rank_bm25 rouge-score transformers bitsandbytes accelerate

import torch, gc, json, pickle, re, time, faiss, os
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
import pyarrow.parquet as pq

torch.manual_seed(42); np.random.seed(42)
DRIVE = Path('/content/drive/MyDrive/hukuk-rag')
TURKISH_LOWER_MAP = str.maketrans("İIÖÜÇŞĞ", "iıöüçşğ")
TURKISH_STOPWORDS = {"bir","bu","da","de","ve","ile","için","olan","olarak","gibi","daha","en","çok","her","kadar","sonra","önce","ise","ya","ne","nasıl","neden","nerede","kim","hangi","o","şu","ben","sen","biz","siz","onlar","mi","mu","mü","mı","dir","dır","dur","dür","tir","tır","tur","tür","ki","ama","ancak","fakat","lakin","veya","yahut","hem","üzere","göre","karşı"}

SYSTEM_PROMPT = (
    "Sen bir Türk hukuku uzmanısın. Soruyu verilen bağlam paragraflarını kullanarak "
    "kısa ve öz şekilde yanıtla (2-3 cümle). İlgili kanun maddelerine atıfta bulun. "
    "Bağlamda bilgi yoksa 'Bu konuda yeterli bilgi bulunamadı' de."
)

def normalize_turkish(text):
    text = text.translate(TURKISH_LOWER_MAP).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
def turkish_tokenize(text):
    text = text.translate(TURKISH_LOWER_MAP).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return [t for t in text.split() if t not in TURKISH_STOPWORDS and len(t) > 1]

# ── Load shared components ──
print("Loading BM25...")
with open(str(DRIVE / 'indexes' / 'bm25.pkl'), 'rb') as f:
    bm25_data = pickle.load(f)
bm25_index_obj = bm25_data['index']; bm25_mapping = bm25_data['mapping']

print("Loading fine-tuned FAISS...")
ft_index = faiss.read_index(str(DRIVE / 'indexes' / 'finetuned' / 'faiss_ft.index'))
with open(str(DRIVE / 'indexes' / 'finetuned' / 'faiss_ft.mapping.pkl'), 'rb') as f:
    ft_ids = pickle.load(f)
ft_index.nprobe = 16

print("Loading chunk texts...")
pf = pq.ParquetFile(str(DRIVE / 'data' / 'processed' / 'chunks_filtered.parquet'))
chunk_text_list = []
for batch in pf.iter_batches(batch_size=100_000, columns=['text']):
    chunk_text_list.extend(batch.column('text').to_pylist())
print(f"  {len(chunk_text_list):,} texts")

print("Loading fine-tuned E5 (CPU)...")
ft_e5 = SentenceTransformer(str(DRIVE / 'models' / 'e5-checkpoints' / 'checkpoint-10000'), device='cpu')

print("Loading Qwen 4-bit (base, no QLoRA)...")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16)
llm.eval()

REPO = Path('/content/hukuk-rag')
if not REPO.exists():
    from google.colab import userdata
    try: token = userdata.get("GITHUB_TOKEN")
    except: token = ""
    os.system(f"git clone https://{token}@github.com/berkay-aktas/hukuk-rag.git {REPO}")
with open(REPO / 'data' / 'gold' / 'gold_test_set.json', encoding='utf-8') as f:
    gold_data = json.load(f)['questions']

print(f"GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB | Gold: {len(gold_data)} questions")

# ── Shared functions ──
def bm25_search(query, k=50):
    tokens = turkish_tokenize(query)
    scores = bm25_index_obj.get_scores(tokens)
    top_idx = np.argsort(scores)[-k:][::-1]
    return [{"chunk_id": bm25_mapping[idx]["chunk_id"], "score": float(scores[idx]),
             "text": bm25_mapping[idx]["text"]} for idx in top_idx if scores[idx] > 0]

def dense_search_ft(query, k=50):
    q_emb = ft_e5.encode([f"query: {query}"], normalize_embeddings=True).astype(np.float32)
    scores, indices = ft_index.search(q_emb, k)
    return [{"chunk_id": ft_ids[idx], "score": float(score), "text": chunk_text_list[idx]}
            for score, idx in zip(scores[0], indices[0]) if idx != -1]

def rrf_merge(d, b, k=60, top_k=10):
    scores = defaultdict(float); best = {}
    for results in [d, b]:
        for rank, r in enumerate(results):
            scores[r["chunk_id"]] += 1.0/(k+rank+1)
            if r["chunk_id"] not in best or r["score"] > best[r["chunk_id"]]["score"]: best[r["chunk_id"]] = r
    ids = sorted(scores.keys(), key=lambda x: scores[x], reverse=True)[:top_k]
    return [{"chunk_id": c, "score": scores[c], "text": best[c]["text"]} for c in ids]

@torch.inference_mode()
def generate_answer(context, question):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":f"Bağlam:\n{context}\n\nSoru: {question}"}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=256, temperature=0.1, top_p=0.9,
        do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def compute_metrics(preds, refs):
    sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    em = sum(1 for p,r in zip(preds,refs) if normalize_turkish(p)==normalize_turkish(r))/len(preds)
    f1s = []
    for pred, ref in zip(preds, refs):
        pt=Counter(normalize_turkish(pred).split()); rt=Counter(normalize_turkish(ref).split())
        if not pt or not rt: f1s.append(0.0); continue
        c=sum((pt&rt).values())
        if c==0: f1s.append(0.0); continue
        p=c/sum(pt.values()); r=c/sum(rt.values()); f1s.append(2*p*r/(p+r))
    rouges=[sc.score(normalize_turkish(ref),normalize_turkish(pred))["rougeL"].fmeasure for pred,ref in zip(preds,refs)]
    return {"exact_match":float(em),"token_f1":float(np.mean(f1s)),"rouge_l":float(np.mean(rouges))}

def run_and_save(rag_fn, config_name, filename):
    preds, refs = [], []
    t0 = time.time()
    for i, item in enumerate(tqdm(gold_data, desc=f"{config_name}")):
        try: a, _ = rag_fn(item["question"]); preds.append(a)
        except: preds.append("")
        refs.append(item["gold_answer"])
        if (i+1)%50==0: print(f"  {i+1}/{len(gold_data)} | {(time.time()-t0)/60:.1f}min")
    elapsed = time.time()-t0
    metrics = compute_metrics(preds, refs)
    print(f"  {config_name}: F1={metrics['token_f1']:.4f}, ROUGE-L={metrics['rouge_l']:.4f} ({elapsed/60:.1f}min)")
    result = {"config": config_name, "metrics": {"generation": metrics},
              "gold_set_size": len(preds), "runtime_minutes": round(elapsed/60, 1)}
    with open(str(DRIVE/'results'/f'{filename}.json'), 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    pred_data = {"config": config_name, "predictions": preds, "references": refs,
        "per_question": [{"question": gold_data[i]["question"], "prediction": preds[i],
         "reference": refs[i], "domain": gold_data[i].get("domain",""),
         "difficulty": gold_data[i].get("difficulty",""),
         "is_answerable": gold_data[i].get("is_answerable", True)} for i in range(len(preds))]}
    with open(str(DRIVE/'results'/f'{filename}_predictions.json'), 'w', encoding='utf-8') as f:
        json.dump(pred_data, f, ensure_ascii=False, indent=2)
    return metrics

print("\nAll loaded. Starting C2 + C3 re-runs...")

# ═══════════════════════════════════════════════════════
# C2: Fine-tuned E5 + BM25 + RRF + Base Qwen
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  CONFIG 2 RE-RUN (fixed prompt)")
print("=" * 60)

def rag_c2(q):
    merged = rrf_merge(dense_search_ft(q, 50), bm25_search(q, 50), 60, 10)
    ctx = "\n\n".join(f"[{i+1}] {r['text']}" for i,r in enumerate(merged))
    return generate_answer(ctx, q), merged

metrics_c2 = run_and_save(rag_c2, "Config 2 - FT Embeddings (fixed prompt)", "config2_finetuned_embeddings")

# ═══════════════════════════════════════════════════════
# C3: Fine-tuned E5 + BM25 + RRF + FT Reranker + Base Qwen
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  CONFIG 3 RE-RUN (fixed prompt)")
print("=" * 60)

print("Loading fine-tuned reranker (CPU)...")
reranker_tok = AutoTokenizer.from_pretrained(str(DRIVE / 'models' / 'reranker-finetuned' / 'model'))
reranker_mdl = AutoModelForSequenceClassification.from_pretrained(
    str(DRIVE / 'models' / 'reranker-finetuned' / 'model'), num_labels=1)
reranker_mdl.eval()

def rerank(query, passages, top_k=10):
    pairs_scores = []
    for i in range(0, len(passages), 8):
        batch = passages[i:i+8]
        inputs = reranker_tok([query]*len(batch), [p["text"] for p in batch],
            return_tensors='pt', truncation=True, max_length=512, padding=True)
        with torch.inference_mode():
            out = reranker_mdl(**inputs).logits.squeeze(-1)
        if out.dim()==0: pairs_scores.append((batch[0], out.item()))
        else: pairs_scores.extend(zip(batch, out.tolist()))
    ranked = sorted(pairs_scores, key=lambda x: x[1], reverse=True)
    return [{"chunk_id":p["chunk_id"],"score":s,"text":p["text"]} for p,s in ranked[:top_k]]

def rag_c3(q):
    merged = rrf_merge(dense_search_ft(q, 50), bm25_search(q, 50), 60, 30)
    reranked = rerank(q, merged, 10)
    ctx = "\n\n".join(f"[{i+1}] {r['text']}" for i,r in enumerate(reranked))
    return generate_answer(ctx, q), reranked

metrics_c3 = run_and_save(rag_c3, "Config 3 - FT Reranker (fixed prompt)", "config3_reranker")

# ═══════════════════════════════════════════════════════
# FINAL COMPARISON TABLE
# ═══════════════════════════════════════════════════════
with open(str(DRIVE / 'results' / 'config1_baseline_225.json')) as f:
    c1 = json.load(f)['metrics']['generation']
with open(str(DRIVE / 'results' / 'config4_qlora.json')) as f:
    c4 = json.load(f)['metrics']['generation']
with open(str(DRIVE / 'results' / 'config5_full.json')) as f:
    c5 = json.load(f)['metrics']['generation']

print("\n" + "=" * 90)
print("  FINAL ABLATION TABLE — ALL CONFIGS WITH FIXED PROMPT (225 questions)")
print("=" * 90)
print(f"{'Metric':<12} {'C1 Base':<12} {'C2 +Emb':<12} {'C3 +Rerank':<12} {'C4 +QLoRA':<12} {'C5 Full':<12}")
print("-" * 90)
print(f"{'EM':<12} {c1['exact_match']:<12.4f} {metrics_c2['exact_match']:<12.4f} {metrics_c3['exact_match']:<12.4f} {c4['exact_match']:<12.4f} {c5['exact_match']:<12.4f}")
print(f"{'Token F1':<12} {c1['token_f1']:<12.4f} {metrics_c2['token_f1']:<12.4f} {metrics_c3['token_f1']:<12.4f} {c4['token_f1']:<12.4f} {c5['token_f1']:<12.4f}")
print(f"{'ROUGE-L':<12} {c1['rouge_l']:<12.4f} {metrics_c2['rouge_l']:<12.4f} {metrics_c3['rouge_l']:<12.4f} {c4['rouge_l']:<12.4f} {c5['rouge_l']:<12.4f}")
print("-" * 90)
print(f"\n  C1→C5 overall: F1 {((c5['token_f1']-c1['token_f1'])/c1['token_f1']*100):+.1f}%")
print("\nAll results saved to Drive.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.1 MB/s eta 0:00:00
Loading BM25...
Loading fine-tuned FAISS...
Loading chunk texts...
  2,496,668 texts
Loading fine-tuned E5 (CPU)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading Qwen 4-bit (base, no QLoRA)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

Config 2 - FT Embeddings (fixed prompt):   0%|          | 0/225 [00:00<?, ?it/s]

  50/225 | 13.2min
  100/225 | 25.7min
  150/225 | 41.2min
  200/225 | 54.2min
  Config 2 - FT Embeddings (fixed prompt): F1=0.1678, ROUGE-L=0.1669 (61.9min)

  CONFIG 3 RE-RUN (fixed prompt)
Loading fine-tuned reranker (CPU)...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Config 3 - FT Reranker (fixed prompt):   0%|          | 0/225 [00:00<?, ?it/s]

  50/225 | 34.8min
